# Step 6b — Multi-corpus feature matrix builder


In [ ]:
# Mount Drive + set PROJECT_ROOT
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
except Exception:
    PROJECT_ROOT = os.path.abspath('.')
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
print('PROJECT_ROOT =', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/text-difficulty-classification


In [ ]:
import argparse
import glob
import os
import sys

import pandas as pd

PROJECT_ROOT = os.environ['PROJECT_ROOT']
GENERIC_DIR  = os.path.join(PROJECT_ROOT, 'outputs', 'static_metrics_generic')
PROMPT_DIR   = os.path.join(PROJECT_ROOT, 'outputs', 'prompt_metrics')
FEATURES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'features_multi')
os.makedirs(FEATURES_DIR, exist_ok=True)

LABEL_COL = 'education_level'

META_COLS = ['education_level', 'source_dataset', 'domain',
             'label_source', 'subject', 'split', 'raw_label', 'text_idx']


In [ ]:
def _impute_hedge(df):
    """Replace -1 values in prompt_* cols with column mean of valid rows."""
    prompt_cols = [c for c in df.columns if c.startswith('prompt_')]
    for c in prompt_cols:
        col = df[c].astype(float)
        if (col < 0).any():
            mean = col[col >= 0].mean()
            df.loc[col < 0, c] = mean if pd.notna(mean) else 0.0
    return df


In [ ]:
def _splits():
    """Discover all split CSVs in outputs/static_metrics_generic/."""
    paths = sorted(glob.glob(os.path.join(GENERIC_DIR, '*_static_generic.csv')))
    return [os.path.basename(p)[:-len('_static_generic.csv')] for p in paths]


In [ ]:
def _list_models(split, suffix):
    """Find every <model> with a prompt-metrics CSV for this split."""
    pattern = os.path.join(PROMPT_DIR, f'{split}_prompt_metrics_*{suffix}.csv')
    out = []
    for p in glob.glob(pattern):
        base = os.path.basename(p)
        # parse: <split>_prompt_metrics_<model>__multi(_extra).csv
        prefix = f'{split}_prompt_metrics_'
        rest = base[len(prefix):-len('.csv')]
        if rest.endswith(suffix):
            model = rest[:-len(suffix)]
            out.append(model)
    return sorted(set(out))


In [ ]:
def _load_static(split):
    path = os.path.join(GENERIC_DIR, f'{split}_static_generic.csv')
    return pd.read_csv(path)


In [ ]:
def _load_prompts(split, model, suffix):
    path = os.path.join(PROMPT_DIR, f'{split}_prompt_metrics_{model}{suffix}.csv')
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    return _impute_hedge(df)


In [ ]:
def _meta_cols(df):
    return df[[c for c in META_COLS if c in df.columns]].copy()


In [ ]:
def _attach_meta(out, meta):
    for c in META_COLS:
        if c in meta.columns:
            out[c] = meta[c].values
    return out


In [ ]:
def _build_for_split(split, suffix):
    static = _load_static(split)
    static_only = static.drop(columns=[c for c in META_COLS if c in static.columns],
                              errors='ignore')
    meta = _meta_cols(static)

    out_static = static_only.copy()
    out_static = _attach_meta(out_static, meta)
    out_static.to_csv(os.path.join(FEATURES_DIR, f'{split}_static.csv'), index=False)

    models = _list_models(split, suffix)
    per_model = {}
    for m in models:
        pm_full = _load_prompts(split, m, suffix)
        if pm_full is None:
            continue
        pm = pm_full[[c for c in pm_full.columns if c.startswith('prompt_')]].copy()
        if len(pm) != len(static_only):
            print(f"[step6b] {split} {m}: row count mismatch — skipping.")
            continue


        if LABEL_COL in pm_full.columns and LABEL_COL in static.columns:
            pm_labels  = pm_full[LABEL_COL].reset_index(drop=True)
            sta_labels = static[LABEL_COL].reset_index(drop=True)
            if not pm_labels.equals(sta_labels):
                n_disagree = int((pm_labels != sta_labels).sum())
                print(f"[step6b] {split} {m}: ABORT — label-row mismatch "
                      f"({n_disagree}/{len(pm_labels)} rows differ). The prompt "
                      f"CSV and static CSV came from different Step 2c runs. "
                      f"Re-sync multi_corpus across Drive/remote and re-run "
                      f"Steps 3b + 5c against the SAME multi_corpus revision.")
                continue
        per_model[m] = pm

        prompts_only = pm.copy()
        prompts_only = _attach_meta(prompts_only, meta)
        prompts_only.to_csv(os.path.join(FEATURES_DIR, f'{split}_prompts_{m}.csv'),
                            index=False)

        fused = pd.concat([static_only.reset_index(drop=True),
                           pm.reset_index(drop=True)], axis=1)
        fused = _attach_meta(fused, meta)
        fused.to_csv(os.path.join(FEATURES_DIR, f'{split}_fused_{m}.csv'), index=False)

    if per_model:
        # all_prompts: every model side-by-side, prefix with model key.
        chunks = []
        for m, pm in per_model.items():
            chunks.append(pm.rename(columns={c: f'{m}__{c}' for c in pm.columns}))
        allp = pd.concat([c.reset_index(drop=True) for c in chunks], axis=1)
        allp = _attach_meta(allp, meta)
        allp.to_csv(os.path.join(FEATURES_DIR, f'{split}_all_prompts.csv'), index=False)

        full = pd.concat([static_only.reset_index(drop=True),
                          allp.drop(columns=META_COLS, errors='ignore').reset_index(drop=True)],
                         axis=1)
        full = _attach_meta(full, meta)
        full.to_csv(os.path.join(FEATURES_DIR, f'{split}_full.csv'), index=False)

    print(f"[step6b] {split}: static cols={static_only.shape[1]} "
          f"models={list(per_model)} ")


In [ ]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--extra', action='store_true',
                    help='Use prompt CSVs with the 73-prompt extra set (suffix __multi_extra).')
    import sys as _sys
    if any(a.endswith('.json') or a.startswith('-f') for a in _sys.argv[1:]):
        args = ap.parse_args([])
    else:
        args = ap.parse_args()
    suffix = '__multi_extra' if args.extra else '__multi'

    splits = _splits()
    if not splits:
        sys.exit(f"[step6b] No generic-static CSVs in {GENERIC_DIR} — run Step 3b first.")

    for s in splits:
        _build_for_split(s, suffix)
    print(f"[step6b] done — files in {FEATURES_DIR}")


In [ ]:
main()


[step6b] ood_onestop: static cols=26 models=['qwen2.5-7b'] 
[step6b] ood_openbookqa: static cols=26 models=['qwen2.5-7b'] 
[step6b] ood_race-high: static cols=26 models=['qwen2.5-7b'] 


/tmp/ipykernel_30672/4138960092.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.04803202134756504' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[col < 0, c] = mean if pd.notna(mean) else 0.0


[step6b] ood_race-middle: static cols=26 models=['qwen2.5-7b'] 
[step6b] test: static cols=26 models=['qwen2.5-7b'] 
[step6b] train: static cols=26 models=['qwen2.5-7b'] 
[step6b] val: static cols=26 models=['qwen2.5-7b'] 
[step6b] done — files in /content/drive/MyDrive/text-difficulty-classification/outputs/features_multi
